In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys
from datetime import datetime
cwd = os.getcwd()
project_root_idx = cwd.split('/').index('ecom_app')
project_root_dir = '/'.join(cwd.split('/')[:project_root_idx+1])
sys.path.append(project_root_dir)


In [3]:
from pyspark.sql import functions as F

In [4]:
from ETL.python.utils.spark_utils import get_spark_session, read_table, write_table, overwrite_table

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/20 14:07:39 WARN Utils: Your hostname, Abhisheks-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.113 instead (on interface en0)
26/05/20 14:07:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 14:07:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/20 14:07:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
spark = get_spark_session()

In [16]:
## Link ETL log json here
last_run_timestamp = '2026-05-20 12:00:00'


In [17]:
# Initializations 
curr_run_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [18]:
customer_table = read_table('ecom_oltp_db', 'customers')
address_table = read_table('ecom_oltp_db', 'address')
dim_customer_table = read_table('ecom_olap_db', 'dim_customer')

# Implementing SCD 2 for customer table

In [19]:
def get_updated_rows_df(df, last_run_timestamp, curr_run_timestamp, updated_at='updated_at', created_at='created_at'):
    last_run_timestamp_col = F.to_timestamp(F.lit(last_run_timestamp), 'yyyy-MM-dd HH:mm:ss')
    curr_run_timestamp_col = F.to_timestamp(F.lit(curr_run_timestamp), 'yyyy-MM-dd HH:mm:ss')
    return df.filter( (F.col(updated_at) >= last_run_timestamp_col ) & (F.col(updated_at) < curr_run_timestamp_col) & (F.col(created_at) < last_run_timestamp_col) )

def get_added_rows_df(df, last_run_timestamp, curr_run_timestamp, created_at='created_at'):
    last_run_timestamp_col = F.to_timestamp(F.lit(last_run_timestamp), 'yyyy-MM-dd HH:mm:ss')
    curr_run_timestamp_col = F.to_timestamp(F.lit(curr_run_timestamp), 'yyyy-MM-dd HH:mm:ss')
    return df.filter( (F.col(created_at) >= last_run_timestamp_col ) & (F.col(created_at) < curr_run_timestamp_col) )

In [20]:
updated_rows_df = get_updated_rows_df(customer_table, last_run_timestamp, curr_run_timestamp)
added_rows_df = get_added_rows_df(customer_table, last_run_timestamp, curr_run_timestamp)

In [21]:
dim_customer_table = dim_customer_table.join(updated_rows_df[['id']], on=[dim_customer_table.customer_id == updated_rows_df.id], how='left')

In [22]:
dim_customer_table = dim_customer_table.withColumn('valid_to', F.when( ((F.col('is_current') == True) & (F.col('id').isNotNull())), F.to_timestamp(F.lit(curr_run_timestamp), 'yyyy-MM-dd HH:mm:ss'))
                                            .otherwise(F.col('valid_to')))\
                                        .withColumn('is_current', F.when( ((F.col('is_current') == True) & (F.col('id').isNotNull())), F.lit(False))
                                            .otherwise(F.col('is_current')))\
                                        .drop('id')

In [23]:
updated_rows_df.show()

+-----+----------+----+-------------+--------------------+--------+-------------------+-------------------+
|   id|address_id|name|mobile_number|               email|  status|         created_at|         updated_at|
+-----+----------+----+-------------+--------------------+--------+-------------------+-------------------+
|16897|      8334| xyz|   8300657742|customer16897@exa...|inactive|2026-04-21 05:24:18|2026-05-20 14:06:51|
+-----+----------+----+-------------+--------------------+--------+-------------------+-------------------+



In [24]:
added_rows_df.show()

+-----+----------+----+-------------+-------------+------+-------------------+-------------------+
|   id|address_id|name|mobile_number|        email|status|         created_at|         updated_at|
+-----+----------+----+-------------+-------------+------+-------------------+-------------------+
|20003|         1| zyx|   7894561290|abc@gmail.com|active|2026-05-20 14:07:14|2026-05-20 14:07:14|
+-----+----------+----+-------------+-------------+------+-------------------+-------------------+



In [25]:
append_dim_cust_table = updated_rows_df.unionByName(added_rows_df)

In [26]:
append_dim_cust_table = append_dim_cust_table.join(address_table, on=[append_dim_cust_table.address_id == address_table.id], how='left').drop(*[address_table.id,address_table.created_at, address_table.updated_at])

In [27]:
append_dim_cust_table = append_dim_cust_table.withColumnRenamed('id', 'customer_id')\
                        .withColumn('address',    F.concat_ws(', ', *['house_number', 'street']))\
                        .withColumn('valid_from', F.to_timestamp(F.lit(curr_run_timestamp), 'yyyy-MM-dd HH:mm:ss'))\
                        .withColumn('valid_to',   F.to_timestamp(F.lit('9999-12-31 23:59:59'), 'yyyy-MM-dd HH:mm:ss'))\
                        .withColumn('is_current', F.lit(True))


In [28]:
dim_customer_table = append_dim_cust_table[dim_customer_table.columns].unionByName(dim_customer_table)

In [29]:
dim_customer_table.count()

4

In [30]:
dim_customer_table.show()

+-----------+--------+-------------+--------------------+-----------------+---------+-----------+-------+--------+-------------------+-------------------+----------+-------------------+
|customer_id|    name|mobile_number|               email|          address|     city|   locality|pincode|  status|         valid_from|           valid_to|is_current|         created_at|
+-----------+--------+-------------+--------------------+-----------------+---------+-----------+-------+--------+-------------------+-------------------+----------+-------------------+
|      20003|     zyx|   7894561290|       abc@gmail.com|      20, MG Road|   Mumbai|    Andheri| 475919|  active|2026-05-20 14:08:17|9999-12-31 23:59:59|      true|2026-05-20 14:07:14|
|      16897|     xyz|   8300657742|customer16897@exa...|     994, SV Road|     Pune|Hitech City| 597875|inactive|2026-05-20 14:08:17|9999-12-31 23:59:59|      true|2026-04-21 05:24:18|
|      16896|abhishek|   8946281833|customer16896@exa...|967, Brigade 

In [32]:
dim_customer_table.show()

ERROR:root:KeyboardInterrupt while sending command.                 (0 + 2) / 2]
Traceback (most recent call last):
  File "/opt/miniconda3/envs/.ecom_app_py310/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/miniconda3/envs/.ecom_app_py310/lib/python3.10/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/miniconda3/envs/.ecom_app_py310/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

26/05/20 14:10:37 ERROR RetryingBlockTransferor: Exception while beginning fetch of 1 outstanding blocks
java.io.IOException: Connecting to /172.21.0.3:39825 failed in the last 4750 ms, fail this connection directly
	at org.apache.spark.network.client.TransportClientFactory.createClient(TransportClientFactory.java:220)
	at org.apache.spark.network.netty.NettyBlockTransferService$$anon$2.createAndStart(NettyBlockTransferService.scala:137)
	at org.apache.spark.network.shuffle.RetryingBlockTransferor.transferAllOutstanding(RetryingBlockTransferor.java:180)
	at org.apache.spark.network.shuffle.RetryingBlockTransferor.start(RetryingBlockTransferor.java:159)
	at org.apache.spark.network.netty.NettyBlockTransferService.fetchBlocks(NettyBlockTransferService.scala:157)
	at org.apache.spark.network.BlockTransferService.fetchBlockSync(BlockTransferService.scala:102)
	at org.apache.spark.storage.BlockManager.fetchRemoteManagedBuffer(BlockManager.scala:1208)
	at org.apache.spark.storage.BlockManage

In [31]:
overwrite_table('ecom_olap_db', 'dim_customer', dim_customer_table)

Table cleared successfully


26/05/20 14:08:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Data write successful
